# V33 Adapter Evaluation on Colab Pro

Run V33 adapter through val_set_950 to diagnose which category caused LB 0.73 disaster.

**Requires**: Colab Pro with **A100 (40GB VRAM)** GPU. T4/V100/L4 will OOM even with INT4.

**Compute units burn**: ~15-25 units
  - Model download (~20 min): negligible compute (download speed bound)
  - vLLM load + inference (~3-5h): ~13 units/hour A100 = 40-65 units... wait that's high
  - Realistic: A100 + INT4 + LoRA on 30B = ~30 tokens/s × 8 seqs = 240 tok/s
  - 950 prompts × ~5000 tokens avg = 4.75M tokens / 240 = **~5.5 hours**

If you want to test quick: change cell 17 to `val = val[:50]` first to validate pipeline.

## 1. Sanity check GPU

In [ ]:
!nvidia-smi | head -15

**Critical**: confirm `A100-SXM4-40GB` or higher. If T4/V100, change runtime: Runtime → Change runtime type → A100.

## 2. Install dependencies

In [ ]:
!pip install -q vllm bitsandbytes peft transformers accelerate huggingface_hub kaggle
!python -c "import vllm; print('vllm:', vllm.__version__)"
!python -c "import bitsandbytes; print('bnb:', bitsandbytes.__version__)"
!python -c "import torch; print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"
# Required: vllm >= 0.6.4 for bitsandbytes + LoRA combination
# If vllm version < 0.6.4, run: !pip install -q --upgrade vllm

## 3. Kaggle credentials

Kaggle 网页现在只给一串 API token（`KGAT_xxx` 形式），不再下载 kaggle.json 文件。
我们自己构造 JSON：

- Username: Kaggle 用户名（如 `exerzitien`）
- API key: 从 kaggle.com/settings → "Create New API Token" 复制的 token 字符串（`KGAT_...`）

In [ ]:
import os
import json
from getpass import getpass

# Username and key (token) — paste both
KAGGLE_USERNAME = input('Kaggle username (e.g. exerzitien): ').strip()
KAGGLE_KEY = getpass('Kaggle API token (KGAT_...): ').strip()

# Construct kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Kaggle creds written. Testing auth via --mine (lists your private + public datasets)...')
# `--mine` lists user's own datasets (includes private). `--user X` only lists public.
# V33 adapter etc. are private, so --user would return "No datasets found" even when auth works.
!kaggle datasets list --mine 2>&1 | head -10

## 4. HuggingFace login (for Nemotron base model)

Get token from https://huggingface.co/settings/tokens (read access).

In [ ]:
from huggingface_hub import login
from getpass import getpass

# Get token from https://huggingface.co/settings/tokens (read scope)
HF_TOKEN = getpass('HF token (hf_...): ').strip()
login(HF_TOKEN)
print('HF login OK')

## 5. Download Nemotron base model (~60GB, 5-20 min in Colab)

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = '/content/nemotron-base'
print('Downloading Nemotron 30B base (~60GB, 15-30 min)...')
# DON'T use allow_patterns — Nemotron has .bin/.safetensors mix + special tokenizer files;
# missing any will make vLLM fail to load. Just download everything.
snapshot_download(
    repo_id='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16',
    local_dir=MODEL_DIR,
    max_workers=4,
)
!ls -lh /content/nemotron-base/ | head -25
print()
!df -h /content | tail -1

## 6. Download V33 adapter (~4GB) + val_set_950

In [ ]:
!mkdir -p /content/adapter-v33 /content/data
!kaggle datasets download exerzitien/nemotron-v33-adapter -p /content/v33-zip --unzip
!unzip -o /content/v33-zip/submission.zip -d /content/adapter-v33
!ls /content/adapter-v33

!kaggle datasets download jiazhuang/nemotron-val-set-950 -p /content/data --unzip
!ls /content/data | grep val_set_950

## 7. Fix adapter_config.json for vLLM compatibility

V33 adapter has Unsloth-specific `target_parameters` field for MoE expert LoRA injection. 
vLLM PEFT may not understand it; set to null and let vLLM load standard target_modules.

In [ ]:
import json
cfg_path = '/content/adapter-v33/adapter_config.json'
with open(cfg_path) as f:
    cfg = json.load(f)
print('Before:')
print('  rank (r):', cfg.get('r'))
print('  target_modules:', cfg.get('target_modules'))
print('  target_parameters:', cfg.get('target_parameters'))

# Keep target_modules but null out target_parameters (Unsloth-specific MoE injection)
# Note: this means MoE expert LoRA weights in safetensors won't be loaded by vLLM.
# This matches Kaggle eval behavior (vLLM only loads keys matching target_modules).
# We DON'T change base_model_name_or_path — vLLM uses the main LLM, doesn't check this.
cfg['target_parameters'] = None
cfg['inference_mode'] = True

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print('\nAfter: target_parameters cleared (vLLM-compatible)')

# Verify adapter weights file present
import os
weights_path = '/content/adapter-v33/adapter_model.safetensors'
assert os.path.exists(weights_path), f'Missing {weights_path}'
print(f'Adapter weights: {os.path.getsize(weights_path) / 1024**3:.2f} GB')

## 8. Run V33 evaluation

In [ ]:
import json
import math
import os
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd

MODEL_DIR = '/content/nemotron-base'
LORA_PATH = '/content/adapter-v33'
VAL_JSONL = '/content/data/val_set_950.jsonl'
OUT_DIR = Path('/content/eval_results')
OUT_DIR.mkdir(exist_ok=True)

# Official metric (post-2026-05-08 spec)
def extract_final_answer(text):
    if text is None:
        return 'NOT_FOUND'
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    for pat in [r'The final answer is:\s*([^\n]+)', r'Final answer is:\s*([^\n]+)']:
        m = re.findall(pat, text, re.IGNORECASE)
        if m:
            return m[-1].strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    return nums[-1] if nums else 'NOT_FOUND'

def verify(stored, predicted):
    s, p = str(stored).strip(), str(predicted).strip()
    try:
        return math.isclose(float(s), float(p), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return p.lower() == s.lower()

def detect_category(prompt):
    if 'secret bit manipulation rule transforms 8-bit binary numbers' in prompt:
        return 'bit_manipulation'
    if 'secret encryption rules are used on text' in prompt:
        return 'cipher'
    if 'secret set of transformation rules is applied to equations' in prompt:
        after_header = prompt.split('Below are a few examples:\n', 1)[1]
        examples_text, rest = after_header.split('\nNow, determine the result for: ', 1)
        question_text = rest.strip()
        if any(c.isdigit() for c in examples_text):
            q_match = re.fullmatch(r'(\d+)(\D)(\d+)', question_text)
            if q_match and re.search(r'\d' + re.escape(q_match.group(2)) + r'\d', examples_text):
                return 'equation_numeric_deduce'
            return 'equation_numeric_guess'
        if len(question_text) == 5:
            q_op = question_text[2]
            for ex_line in examples_text.strip().splitlines():
                inp = ex_line.split(' = ')[0].strip()
                if len(inp) == 5 and inp[2] == q_op:
                    return 'cryptarithm_deduce'
        return 'cryptarithm_guess'
    if 'gravitational constant has been secretly changed' in prompt:
        return 'gravity'
    if 'converted into a different numeral system' in prompt:
        return 'numeral'
    if 'secret unit conversion is applied to measurements' in prompt:
        return 'unit_conversion'
    return 'unknown'

# Load val_950
val = []
with open(VAL_JSONL) as f:
    for line in f:
        line = line.strip()
        if line:
            val.append(json.loads(line))
for item in val:
    item['category'] = detect_category(item['prompt'])
print(f'val_set_950: {len(val)} rows')
print(f'Categories: {dict(sorted(Counter(item["category"] for item in val).items()))}')

In [ ]:
# Initialize vLLM with INT4 (bitsandbytes) quantization
# A100 40GB tight budget:
#   INT4 base ~15GB + LoRA ~0.5GB + activations ~5GB + KV cache 4 seqs × 8192 ≈ 8GB
#   Total ~28GB / 40GB → comfortable margin
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

print('Loading vLLM (5-15 min — base model needs INT4 conversion on-the-fly)...')
t0 = time.time()
llm = LLM(
    model=MODEL_DIR,
    quantization='bitsandbytes',
    load_format='bitsandbytes',
    tensor_parallel_size=1,
    max_num_seqs=4,                  # 4 is safer than 8 on A100 40GB
    gpu_memory_utilization=0.88,     # leave headroom
    dtype='bfloat16',
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=32,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)
print(f'  Loaded in {(time.time() - t0)/60:.1f} min')
!nvidia-smi | grep -E "MiB / |MiB /" | head -1

In [ ]:
# Prepare prompts
tokenizer = llm.get_tokenizer()
prompts = []
for item in val:
    user_content = item['prompt'] + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    try:
        p = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_content}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True,
        )
    except Exception:
        p = user_content
    prompts.append(p)
print(f'Prepared {len(prompts)} prompts')

# OPTIONAL FAST TEST: uncomment to validate pipeline on small subset first
# prompts = prompts[:50]
# val = val[:50]
# print(f'  (truncated to first {len(prompts)} for fast smoke test)')

# Run inference
print(f'Running inference on {len(prompts)} prompts (estimated {len(prompts)*30/240/60:.1f}h on A100 INT4)...')
t0 = time.time()
outputs = llm.generate(
    prompts,
    sampling_params=SamplingParams(temperature=0.0, max_tokens=7680),
    lora_request=LoRARequest('v33', 1, LORA_PATH),
)
elapsed = max(time.time() - t0, 1.0)
print(f'Inference done in {elapsed/60:.1f} min ({len(prompts)/elapsed:.2f} prompts/s)')

In [ ]:
# Score + report per-category accuracy
results = []
for item, output in zip(val, outputs):
    raw = output.outputs[0].text
    pred = extract_final_answer(raw)
    gold = str(item['answer'])
    results.append({
        'id': item['id'],
        'category': item['category'],
        'gold': gold,
        'pred': pred,
        'correct': verify(gold, pred),
        'raw_tail': raw[-200:],
    })

df = pd.DataFrame(results)
df.to_csv(OUT_DIR / 'v33_results.csv', index=False)

print('=' * 50)
print(f'V33 Overall: {df["correct"].sum()}/{len(df)} ({100*df["correct"].mean():.2f}%)')
print('=' * 50)
print(f'{"category":30s} {"correct":>8s} {"total":>8s} {"pct":>8s}')
for cat in sorted(df['category'].unique()):
    sub = df[df['category'] == cat]
    c = sub['correct'].sum()
    t = len(sub)
    print(f'{cat:30s} {c:>8d} {t:>8d} {100*c/t:>7.2f}%')

# Save mistakes per category
mistakes_dir = OUT_DIR / 'v33_mistakes'
mistakes_dir.mkdir(exist_ok=True)
for cat in df['category'].unique():
    wrong = df[(df['category'] == cat) & (~df['correct'])]
    if len(wrong) > 0:
        wrong.to_csv(mistakes_dir / f'{cat}.csv', index=False)

print(f'\nSaved to {OUT_DIR}/')
print('Send back the table above + check /content/eval_results/v33_mistakes/ for category-specific failure examples.')

## 9. Download results to local

Download via Files panel (left sidebar) or:

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/v33_eval_results', 'zip', '/content/eval_results')
files.download('/content/v33_eval_results.zip')